# ==============================================================================
# SEMINAR WORK: DBSCAN FOR DETECTING SPATIAL CLUSTERS IN GEOGRAPHIC DATA
# Advanced Tourism Corridors & Regional Border Sensitivity Engine
# ==============================================================================

In [92]:

# 1. Production Pipeline Initializations & Modular Imports

# %pip install matplotlib pandas numpy folium scikit-learn

import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium
from folium.plugins import MiniMap
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import pdist, squareform

# Append parent directory modularly to access local data frameworks
sys.path.append(os.path.abspath("../"))
from src.data_processing import load_and_validate_data, get_radial_coordinates
from src.clustering_engine import run_spatial_dbscan

# Portable Path Configuration
LOCAL_DATA_PATH = "../data/heritage_sites_bih.csv"
df = load_and_validate_data(LOCAL_DATA_PATH)
X_rad = get_radial_coordinates(df)

print(f"Pipeline Executed: Loaded {df.shape[0]} cultural sites across {df.shape[1]} asset variables.")

Pipeline Executed: Loaded 272 cultural sites across 22 asset variables.


In [97]:
# Cell 2: Multi-Scale Hyperparameter Influence Analysis (Seminar Core)
print(" Executing Optimized Multi-Scale DBSCAN Engines...")

EARTH_RADIUS_KM = 6371.009

# 1. Scale A: Micro-Regional Density
EPS_MICRO_KM = 25.0
MIN_PTS_MICRO = 4
eps_micro_rad = EPS_MICRO_KM / EARTH_RADIUS_KM

db_micro = DBSCAN(eps=eps_micro_rad, min_samples=MIN_PTS_MICRO, metric='haversine')
df['cluster_micro'] = db_micro.fit_predict(X_rad)

# 2. Scale B: Macro-Regional Continuous Corridors
EPS_MACRO_KM = 45.0
MIN_PTS_MACRO = 3
eps_macro_rad = EPS_MACRO_KM / EARTH_RADIUS_KM

db_macro = DBSCAN(eps=eps_macro_rad, min_samples=MIN_PTS_MACRO, metric='haversine')
df['cluster_macro'] = db_macro.fit_predict(X_rad)

print(f" Clustering Complete!")
print(f"-> Micro Scale Found: {len(set(df['cluster_micro'])) - (1 if -1 in df['cluster_micro'].values else 0)} clusters.")
print(f"-> Macro Scale Found: {len(set(df['cluster_macro'])) - (1 if -1 in df['cluster_macro'].values else 0)} clusters.")

 Executing Optimized Multi-Scale DBSCAN Engines...
 Clustering Complete!
-> Micro Scale Found: 7 clusters.
-> Macro Scale Found: 1 clusters.


In [98]:
# Cell 3: Geostatistical Diagnostic: Automated Border Proximity Weighting
print(" Commencing Border Proximity & Edge-Truncation Analysis...")

# Approximate bounding polygon vectors for Bosnia & Herzegovina to calculate edge proximity
# Standardizing geographic limits: North: ~45.27, South: ~42.56, West: ~15.72, East: ~19.62
def calculate_border_proximity_stress(lat, lon):
    """Measures minimum degrees of freedom to nearest international administrative boundary."""
    dist_n = abs(45.27 - lat)
    dist_s = abs(lat - 42.56)
    dist_w = abs(lon - 15.72)
    dist_e = abs(19.62 - lon)
    return min(dist_n, dist_s, dist_w, dist_e)

df['border_distance_deg'] = df.apply(lambda row: calculate_border_proximity_stress(row['latitude'], row['longitude']), axis=1)

# Flag edge cases where DBSCAN density is highly susceptible to administrative truncation
BORDER_THRESHOLD_DEG = 0.25
df['is_border_truncated'] = df['border_distance_deg'] <= BORDER_THRESHOLD_DEG

 Commencing Border Proximity & Edge-Truncation Analysis...


In [99]:
# Cell 4:  Seminar Presentation Map Dashboard
print(" Rendering presentation canvas...")

presentation_map = folium.Map(
    location=[44.15, 17.80], 
    zoom_start=8, 
    tiles="CartoDB dark_matter",
    control_scale=True
)

# Vivid high-contrast neon palette for dark mode visibility
palette = ['#00d2ff', '#70ff00', '#ff0076', '#ffb300', '#a000ff', '#00ffaa', '#ff5722', '#eccc68']

micro_layer = folium.FeatureGroup(name="🔍 Micro Scale (Eps: 25km, MinPts: 4)", show=False)
macro_layer = folium.FeatureGroup(name="🗺️ Macro Scale (Eps: 45km, MinPts: 3)", show=True)

fun_fact_col = 'fun_fact' if 'fun_fact' in df.columns else 'fun_facts'

for idx, row in df.iterrows():
    lat, lon = row['latitude'], row['longitude']
    c_micro = row['cluster_micro']
    c_macro = row['cluster_macro']
    border_status = "⚠️ Edge Truncation Alert" if row['is_border_truncated'] else "✅ Stable Inland Vector"
    
    # --- Micro Layer Processing ---
    # Noise is clean distinct white, clusters get bright hex colors
    color_micro = '#ffffff' if c_micro == -1 else palette[c_micro % len(palette)]
    fill_op_micro = 0.2 if c_micro == -1 else 0.8
    status_micro = "SPATIAL NOISE" if c_micro == -1 else f"LOCAL CORRIDOR {c_micro + 1}"
    
    popup_micro = f"""
    <div style="font-family: 'Segoe UI', sans-serif; width: 260px; background: #1e272e; color: #f5f6fa; padding: 10px; border-radius: 6px; border-left: 4px solid {color_micro};">
        <span style="font-size: 9px; font-weight: 800; color: {color_micro};">{status_micro}</span>
        <h4 style="margin: 4px 0; color: #fff;">{row['name']}</h4>
        <p style="margin: 0; font-size: 11px; color: #dcdde1;"><b>Context:</b> {border_status}</p>
    </div>
    """
    
    folium.CircleMarker(
        location=[lat, lon], radius=5, popup=folium.Popup(popup_micro, max_width=300),
        color=color_micro, fill=True, fill_color=color_micro, fill_opacity=fill_op_micro, weight=1.5
    ).add_to(micro_layer)
    
    # --- Macro Layer Processing ---
    color_macro = '#ffffff' if c_macro == -1 else palette[c_macro % len(palette)]
    fill_op_macro = 0.2 if c_macro == -1 else 0.8
    status_macro = "SPATIAL NOISE" if c_macro == -1 else f"NATIONAL CORRIDOR {c_macro + 1}"
    
    popup_macro = f"""
    <div style="font-family: 'Segoe UI', sans-serif; width: 280px; background: #151e24; color: #f5f6fa; padding: 12px; border-radius: 6px; border-left: 4px solid {color_macro};">
        <span style="font-size: 9px; font-weight: 800; color: {color_macro};">{status_macro}</span>
        <h4 style="margin: 4px 0 6px 0; color: #fff;">{row['name']}</h4>
        <p style="margin: 0 0 6px 0; font-size: 11px; color: #dcdde1;"><b>Region:</b> {row['location']}<br><b>Context:</b> {border_status}</p>
        <p style="font-size: 11px; font-style: italic; color: #f5f6fa;">"{row['description']}"</p>
        <div style="background: rgba(255,255,255,0.06); padding: 6px; border-radius: 4px; font-size: 11px; color: #00d2ff;">
            💡 <b>Fact:</b> {row[fun_fact_col]}
        </div>
    </div>
    """
    
    folium.CircleMarker(
        location=[lat, lon], radius=6, popup=folium.Popup(popup_macro, max_width=320),
        color=color_macro, fill=True, fill_color=color_macro, fill_opacity=fill_op_macro, weight=2,
        tooltip=row['name']
    ).add_to(macro_layer)

micro_layer.add_to(presentation_map)
macro_layer.add_to(presentation_map)
folium.LayerControl(collapsed=False).add_to(presentation_map)
MiniMap(toggle_display=True, tile_layer="CartoDB positron", position="bottomleft").add_to(presentation_map)

presentation_map.save("bih_integrated_density_dashboard.html")
presentation_map

 Rendering presentation canvas...


In [96]:
# Cell 5: Advanced Silhouette Analysis Validation Pipeline
from sklearn.metrics import silhouette_score

# Range of epsilon search parameters to test (in kilometers)
eps_range_km = [10.0, 15.0, 20.0, 25.0, 30.0, 40.0, 50.0]
results = []

print("🔬 Running Parameter Grid Scan & Silhouette Evaluation...")
print(f"{'Epsilon (KM)':<15}{'Clusters Found':<20}{'Silhouette Score':<20}")
print("-" * 55)

for eps_km in eps_range_km:
    # Run model instance
    test_db = run_spatial_dbscan(X_rad, eps_km=eps_km, min_samples=4)
    labels = test_db.labels_
    
    # Calculate metrics only if the model found valid clusters (excluding noise)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    
    if n_clusters > 1:
        # Crucial requirement: silhouette score MUST compute using the same spatial metric
        score = silhouette_score(X_rad, labels, metric='haversine')
        results.append((eps_km, score, n_clusters))
        print(f"{eps_km:<15}{n_clusters:<20}{score:.4f}")
    else:
        print(f"{eps_km:<15}{n_clusters:<20}N/A (Insufficient Clusters)")

# Automatically isolate and highlight the mathematically optimal choice
if results:
    best_eps, best_score, best_cl = max(results, key=lambda x: x[1])
    print(f"\n MATHEMATICAL PROOF COMPLETE:")
    print(f"-> Peak Spatial Silhouette Density occurs at Epsilon = {best_eps} KM (Score: {best_score:.4f}).")
    print(f"-> We will use this exact metric in documentation to justify structural parameters!")

🔬 Running Parameter Grid Scan & Silhouette Evaluation...
Epsilon (KM)   Clusters Found      Silhouette Score    
-------------------------------------------------------
10.0           18                  0.2230
15.0           16                  0.3249
20.0           10                  0.0126
25.0           7                   0.0374
30.0           3                   0.0177
40.0           1                   N/A (Insufficient Clusters)
50.0           1                   N/A (Insufficient Clusters)

 MATHEMATICAL PROOF COMPLETE:
-> Peak Spatial Silhouette Density occurs at Epsilon = 15.0 KM (Score: 0.3249).
-> We will use this exact metric in documentation to justify structural parameters!
